# 🤖 Lab 3: Talk to the Machine — LLMs & Prompt Engineering

All day you learned what's inside the box: tokens → embeddings → attention → next-token prediction. Now you get the keys to the box. 🔑

In this lab you'll call a real LLM through an API and run experiments on it like the scientist you now are:

## 📚 The plan
1. 🔌 Your first API call
2. 🎭 Change the model's personality with ONE sentence (system prompts)
3. 🌡️ Break its consistency with temperature
4. 🛠️ Prompt engineering: the techniques that make LLMs actually useful
5. 🧱 Hit the wall — the question no LLM can answer (tomorrow's cliffhanger)

🔑 **You need an API key** —  NEVER paste keys into code cells or share notebooks containing them!

### 🛠️ Setup

We use **OpenRouter** — one API that gives access to many models. The cell below asks for your key **securely** (it won't be saved in the notebook).

In [ ]:
!pip install openai -q

from openai import OpenAI
from getpass import getpass

api_key = getpass("🔑 Paste your OpenRouter API key (input stays hidden): ")

# client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
client = OpenAI(api_key=api_key)

In [ ]:
MODEL = "gpt-5.4-nano"

# OPEN ROUTER ENDPOINTS
# MODEL = "openrouter/free"
# MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
# MODEL = "openai/gpt-oss-120b:free"
# MODEL = "meta-llama/llama-3.1-8b-instruct"

def ask(prompt, system=None, temperature=0.7, timeout=15):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        timeout=timeout
    )
    return response.choices[0].message.content

print("🤖 Connected! Let's talk to the machine.")

### 🔌 Section 1: First Contact

In [ ]:
print(ask("In two sentences, what is machine learning?"))

That's it — you just sent tokens to a Transformer with billions of weights, and it predicted the reply token by token. Remember: it is doing **next-token prediction**, nothing more. Everything you'll see below follows from that one fact.

### 🎭 Section 2: The System Prompt — Personality in One Sentence

The **system prompt** is a hidden instruction that shapes the whole conversation. Same question, three personalities:

In [ ]:
question = "Why is the sky blue?"

personas = {
    "🏴‍☠️ Pirate":        "You are a dramatic pirate. Answer everything in pirate speak.",
    "👶 For a 5-year-old": "Explain everything as if talking to a 5-year-old, in 2 short sentences.",
    "🎓 Strict professor": "You are a strict physics professor. Be precise, formal, and use correct terminology. Maximum 3 sentences.",
}

for name, system in personas.items():
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(ask(question, system=system))

### 🎯 Mini Exercise 2.1
Write your own persona and ask it something. Ideas: a football commentator 📣, a poet who only answers in 4-line poems, a grandmother giving life advice. What's the weirdest persona that still gives CORRECT information?

In [ ]:
my_persona = ""   # TO-DO
my_question = ""  # TO-DO
print(ask(my_question, system=my_persona))

### 🌡️ Section 3: Temperature — The Chaos Knob

From theory: the model predicts a **probability for every possible next token**. Temperature controls how it picks:
- **temperature = 0** → always take the most likely token (deterministic, consistent)
- **high temperature** → sample adventurously (creative... or chaotic)

Let's prove it: same prompt, 3 runs each.

In [ ]:
prompt = "Complete this sentence with exactly 5 more words: 'The robot walked into the'"

print("🧊 TEMPERATURE = 0 (three runs):")
for run in range(3):
    print(f"   {run+1}. {ask(prompt, temperature=0)}")

print("\n🔥 TEMPERATURE = 1.3 (three runs):")
for run in range(3):
    print(f"   {run+1}. {ask(prompt, temperature=1.3)}")

**Observe:** temperature 0 gives (nearly) the same answer every run; 1.3 gives different ones. 💡 Rule of thumb: **low temperature** for facts, code, and extraction; **higher temperature** for brainstorming and creative writing.

### 🛠️ Section 4: Prompt Engineering — Four Techniques That Matter

The model's output quality depends heavily on the input's quality. These four techniques cover 90% of real-world prompting.

In [ ]:
# TECHNIQUE 1: Be specific. Vague in = vague out.
vague    = "Tell me about exercise"
specific = ("Write a 3-point beginner gym plan for someone who has 45 minutes, "
            "3 days a week, and no equipment at home. Use bullet points.")

print("😴 VAGUE PROMPT:\n", ask(vague)[:300], "...\n")
print("🎯 SPECIFIC PROMPT:\n", ask(specific))

In [ ]:
# TECHNIQUE 2: Few-shot — SHOW the model the pattern instead of describing it
few_shot = """Classify the sentiment. Follow the exact format of the examples.

Review: "The food was incredible" → POSITIVE
Review: "Worst service of my life" → NEGATIVE
Review: "It was fine I guess" → NEUTRAL
Review: "I waited an hour and the order was wrong" →"""

print(ask(few_shot, temperature=0))

In [ ]:
# TECHNIQUE 3: Chain of thought — ask it to REASON before answering
puzzle = "A battery pack and charger cost 110 SAR together. The pack costs 100 SAR more than the charger. How much is the charger?"

print("⚡ DIRECT (often falls in the trap):")
print(ask(puzzle + " Answer with just the number, no explanation.", temperature=0))

print("\n🧠 STEP BY STEP:")
print(ask(puzzle + " Think step by step, then give the final answer.", temperature=0))

# (Correct answer: 5 SAR. The intuitive-but-wrong answer is 10.)

In [ ]:
# TECHNIQUE 4: Structured output — demand a format machines can use
extract = """Extract the details from this message as JSON with keys: name, city, request.
Reply with ONLY the JSON, nothing else.

Message: "Hi, this is Sara from Dammam, I'd like to reschedule my appointment to Sunday."
"""

reply = ask(extract, temperature=0)
print(reply)

import json
data = json.loads(reply.replace('```json', '').replace('```', '').strip())
print("\n✅ Parsed into a Python dict:", data['name'], '—', data['city'])
# THIS is how LLMs plug into real software: text in, structured data out.

### 🎯 Mini Exercise 4.1 — Prompt Battle ⚔️
Pick a task (e.g., "summarize this paragraph", "generate 5 quiz questions about CNNs", "write a product description"). Write a LAZY prompt and an ENGINEERED prompt (specific + format + example). Compare outputs. Which techniques improved it most?

In [ ]:
# TO-DO: your lazy vs engineered prompt battle


### 🧱 Section 5: The Wall — What the Box Cannot Do

One last experiment. Ask it things that require **knowing the present**, **calculating precisely**, or **taking action**:

In [ ]:
print("🌦️ Question 1:", "What is the weather in Dammam RIGHT NOW?")
print(ask("What is the weather in Dammam right now? Give me the actual current temperature.", temperature=0)[:400])

print("\n🧮 Question 2:", "What is 847,293 × 652?")
print(ask("What is 847293 * 652? Reply with only the number.", temperature=0))
print("   (The correct answer is:", 847293 * 652, "— did it get it right? 👀)")

**What just happened:** the model either admitted it can't know the weather, or worse — made something up. And the multiplication? It *predicted plausible-looking digits* instead of calculating. A brain with **no eyes, no calculator, no hands.**

Tomorrow, we fix all three. We give the LLM tools: retrieval (eyes 👀), functions (hands 🛠️), and a loop that lets it think→act→observe. That's **Agentic AI** — see you at Day 5. 🚀

## The GOLDEN Question 🏆

At temperature 0, the model gave the same answer every time. At 1.3, it got creative.

**Using ONE idea from today's theory — "the model predicts a probability for every possible next token" — explain what temperature is actually doing. Then explain why hallucinations can happen even at temperature 0.** 🤔